# 08 — 5-Fold Stratified Ensemble (APTOS Fine-Tuning)

Train 5 separate models on 5 different stratified splits of APTOS, starting
from the **same Stage 1 (EyePACS) checkpoint**.

**Why 5-Fold Ensemble?**
- With only ~3,600 APTOS images, a single train/validation split is vulnerable to split bias
- By training 5 separate models and averaging their softmax probabilities at inference,
  we dramatically reduce variance from individual fold quirks
- QWK's quadratic weighting means extreme errors are *heavily* penalised — ensembling suppresses these
- This approach directly improves both **accuracy** and **QWK**

**Prerequisites:**
1. Stage 1 EyePACS pre-training completed (`best_model_s1_phase3.pth`)
2. (Optional) EyePACS label cleaning done (`scripts/clean_eyepacs_labels.py`)
3. APTOS dataset downloaded

**Training uses the same AMP dtype as Stage 1 (`config.AMP_DTYPE` / `Trainer`).**

In [ ]:
import importlib
import os
import sys
import time

IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/diabetic_retinopathy"
    %pip install -q timm albumentations kaggle scipy
else:
    PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.config
importlib.reload(src.config)
from src import config
config.ensure_dirs()

In [ ]:
import numpy as np
import pandas as pd
import torch

from src.augmentations import get_train_transforms, get_valid_transforms
from src.data_manager import ensure_aptos_dataset
from src.dataset import (
    compute_class_weights,
    create_dataloaders,
    load_and_split_kfold,
    preprocess_images_to_cache,
)
from src.evaluate import (
    compute_metrics,
    optimize_qwk_thresholds,
    apply_qwk_thresholds,
    plot_training_curves,
    predict,
    predict_kfold_ensemble,
    predict_with_tta,
)
from src.model import (
    ModelEMA,
    build_model,
    count_parameters,
    unfreeze_backbone,
)
from src.trainer import Trainer
from src.utils import get_device, log_gpu_memory, seed_everything, write_run_manifest

seed_everything()
device = get_device()
log_gpu_memory("Initial")

write_run_manifest(
    config.RUN_MANIFEST_PATH,
    {"notebook_stage": "08_kfold_start", "AMP_DTYPE": config.AMP_DTYPE},
    merge=True,
)

## 1. Download & Prepare APTOS Dataset

In [ ]:
# Download APTOS if not present (auto-downloads from Kaggle)
ensure_aptos_dataset()

# K-Fold split: hold out fixed test set, then split remainder into 5 folds
test_df, folds = load_and_split_kfold(dataset_name="aptos")

print(f"Test set: {len(test_df)} images (held out, same for all folds)")
print(f"Number of folds: {len(folds)}")
for i, (train_df, valid_df) in enumerate(folds):
    print(f"  Fold {i}: train={len(train_df)}, valid={len(valid_df)}")

# Save test set for later evaluation
test_csv_path = os.path.join(config.OUTPUTS_DIR, "test_split_kfold_aptos.csv")
test_df.to_csv(test_csv_path, index=False)
print(f"\nTest set saved to: {test_csv_path}")

## 2. Preprocessing Cache

In [ ]:
# Build/verify preprocessed cache for all APTOS images
# We cache using the first fold's train+valid and the test set
# (subsequent folds reuse the same cached images)
print("Building APTOS preprocessed cache...")
train_df_0, valid_df_0 = folds[0]
preprocess_images_to_cache(train_df_0, dataset_name="aptos")
preprocess_images_to_cache(valid_df_0, dataset_name="aptos")
preprocess_images_to_cache(test_df, dataset_name="aptos")
print("Cache ready.")

## 3. K-Fold Training

Each fold:
1. Builds a fresh model and loads Stage 1 (EyePACS) weights
2. Fully unfreezes the backbone for fine-tuning
3. Trains with the same hyperparameters as the single-model fine-tuning
4. Saves the best checkpoint per fold

| Setting | Value |
|---------|-------|
| LR (backbone) | config.STAGE2_LR |
| LR (head) | 10x backbone |
| Weight decay | config.STAGE2_WEIGHT_DECAY |
| Epochs | config.STAGE2_EPOCHS |
| Mixup | No |
| EMA | Yes |

In [ ]:
# --- Configuration ---
STAGE1_CHECKPOINT = os.path.join(config.CHECKPOINTS_DIR, "best_model_s1_phase3.pth")
N_FOLDS = config.KFOLD_N_SPLITS
START_FOLD = 0  # Change to resume from a specific fold

img_size = config.STAGE2_IMG_SIZE
train_tfm = get_train_transforms(img_size)
valid_tfm = get_valid_transforms(img_size)

fold_kappas = []
total_t0 = time.time()

for fold_idx in range(START_FOLD, N_FOLDS):
    fold_train_df, fold_valid_df = folds[fold_idx]

    print(f"\n{'='*60}")
    print(f"  FOLD {fold_idx + 1}/{N_FOLDS}")
    print(f"  Train: {len(fold_train_df)} | Valid: {len(fold_valid_df)}")
    print(f"{'='*60}")

    # --- Build fresh model and load Stage 1 weights ---
    seed_everything()
    model = build_model(pretrained=True).to(device)

    if os.path.isfile(STAGE1_CHECKPOINT):
        ckpt = torch.load(STAGE1_CHECKPOINT, map_location=device, weights_only=False)
        if "ema_state_dict" in ckpt:
            print("  Loading EMA weights from Stage 1 checkpoint...")
            model.load_state_dict(ckpt["ema_state_dict"])
        else:
            print("  Loading model weights from Stage 1 checkpoint...")
            model.load_state_dict(ckpt["model_state_dict"])
        metrics = ckpt.get("metrics", {})
        print(f"  Stage 1 metrics: QWK={metrics.get('kappa', 'N/A')}")
    else:
        print(f"  WARNING: Stage 1 checkpoint not found at {STAGE1_CHECKPOINT}")
        print("  Using ImageNet pretrained weights instead.")

    # --- Full unfreeze for fine-tuning ---
    unfreeze_backbone(model, from_block=None)
    params = count_parameters(model)
    print(f"  Trainable parameters: {params['trainable']:,} ({params['trainable_pct']:.1f}%)")

    # --- EMA ---
    ema = ModelEMA(model, decay=config.STAGE2_EMA_DECAY) if config.USE_EMA else None
    if ema:
        print(f"  EMA enabled (decay={config.STAGE2_EMA_DECAY})")

    # --- Class weights ---
    class_weights = compute_class_weights(fold_train_df["level"].values)

    # --- DataLoaders ---
    loaders = create_dataloaders(
        train_df=fold_train_df,
        valid_df=fold_valid_df,
        dataset_name="aptos",
        train_transform=train_tfm,
        valid_transform=valid_tfm,
        batch_size=config.STAGE2_BATCH_SIZE,
        img_size=img_size,
    )

    # --- Trainer ---
    trainer = Trainer(
        model=model,
        device=device,
        class_weights=class_weights if config.USE_CLASS_WEIGHTED_LOSS else None,
        ema=ema,
    )

    # --- Train ---
    completed = trainer.fit_phase(
        phase=f"fold_{fold_idx}",
        train_loader=loaders["train"],
        valid_loader=loaders["valid"],
        epochs=config.STAGE2_EPOCHS,
        lr=config.STAGE2_LR,
        weight_decay=config.STAGE2_WEIGHT_DECAY,
        warmup_epochs=config.STAGE2_WARMUP_EPOCHS,
        use_mixup=config.USE_MIXUP,
        use_amp=True,
    )

    status = "completed" if completed else "early-stopped"
    print(f"\n  Fold {fold_idx + 1} {status}")
    print(f"  Best validation QWK: {trainer.best_kappa:.4f}")
    fold_kappas.append(trainer.best_kappa)

    # --- Copy best checkpoint to kfold directory ---
    import shutil
    src_path = os.path.join(config.CHECKPOINTS_DIR, f"best_model_fold_{fold_idx}.pth")
    dst_path = os.path.join(config.KFOLD_CHECKPOINTS_DIR, f"best_model_fold_{fold_idx}.pth")
    if os.path.isfile(src_path):
        os.makedirs(config.KFOLD_CHECKPOINTS_DIR, exist_ok=True)
        shutil.copy2(src_path, dst_path)
        print(f"  Saved fold checkpoint: {dst_path}")

    # --- Save training history ---
    trainer.save_history(
        os.path.join(config.LOGS_DIR, f"training_history_fold_{fold_idx}.json")
    )

    # --- Cleanup GPU memory ---
    del model, trainer, ema
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

total_elapsed = time.time() - total_t0

# --- Summary ---
print(f"\n{'='*60}")
print("  K-FOLD TRAINING COMPLETE")
print(f"{'='*60}")
print(f"  Total time: {total_elapsed:.0f}s ({total_elapsed/60:.1f} min)")
print("  Per-fold best QWK:")
for i, kappa in enumerate(fold_kappas):
    print(f"    Fold {i}: {kappa:.4f}")
if fold_kappas:
    print(f"  Mean QWK:  {np.mean(fold_kappas):.4f}")
    print(f"  Std QWK:   {np.std(fold_kappas):.4f}")
print(f"\n  Checkpoints saved to: {config.KFOLD_CHECKPOINTS_DIR}")

## 4. K-Fold Ensemble Evaluation

Load all 5 fold models, run TTA on each, average probabilities across folds,
then apply QWK threshold optimisation.

In [ ]:
# Build a model architecture (weights will be loaded from checkpoints)
eval_model = build_model(pretrained=False).to(device)

print("Running K-Fold Ensemble Evaluation on test set...")
print("  TTA: enabled")
print("  Multi-scale TTA: enabled")
print(f"  Folds: {config.KFOLD_N_SPLITS}")

y_true_test, y_pred_test, y_proba_test = predict_kfold_ensemble(
    eval_model,
    test_df,
    "aptos",
    device,
    use_tta=True,
    use_multiscale_tta=True,
)

# --- QWK threshold optimization ---
# First get validation predictions (use last fold's validation set for threshold fitting)
last_fold_valid_df = folds[-1][1]
y_true_val, y_pred_val, y_proba_val = predict_kfold_ensemble(
    eval_model,
    last_fold_valid_df,
    "aptos",
    device,
    use_tta=True,
    use_multiscale_tta=True,
)

# Optimise thresholds on validation set
print("\nOptimizing QWK thresholds on validation set...")
thresholds = optimize_qwk_thresholds(y_true_val, y_proba_val)
print(f"  Optimized thresholds: {thresholds}")

# Apply thresholds to test set
y_pred_optimised = apply_qwk_thresholds(y_proba_test, thresholds)

# --- Final metrics ---
print("\n" + "="*60)
print("  RESULTS: K-Fold Ensemble + TTA + Threshold Optimization")
print("="*60)

# Without threshold optimization
metrics_raw = compute_metrics(y_true_test, y_pred_test, y_proba_test)
print("\n  Without threshold optimization:")
print(f"    Accuracy: {metrics_raw['acc']:.4f}")
print(f"    QWK:      {metrics_raw['kappa']:.4f}")

# With threshold optimization
metrics_opt = compute_metrics(y_true_test, y_pred_optimised, y_proba_test)
print("\n  With threshold optimization:")
print(f"    Accuracy: {metrics_opt['acc']:.4f}")
print(f"    QWK:      {metrics_opt['kappa']:.4f}")

# Save to manifest
write_run_manifest(
    config.RUN_MANIFEST_PATH,
    {
        "notebook_stage": "08_kfold_complete",
        "kfold_eval": {
            "n_folds": config.KFOLD_N_SPLITS,
            "fold_kappas": fold_kappas,
            "mean_fold_kappa": float(np.mean(fold_kappas)) if fold_kappas else None,
            "test_acc_raw": float(metrics_raw['acc']),
            "test_qwk_raw": float(metrics_raw['kappa']),
            "test_acc_optimised": float(metrics_opt['acc']),
            "test_qwk_optimised": float(metrics_opt['kappa']),
            "thresholds": thresholds.tolist() if hasattr(thresholds, 'tolist') else list(thresholds),
        },
    },
    merge=True,
)
print("\nResults saved to run manifest.")